# 환율 데이터 수집 및 테이블 생성

한국수출입은행 환율 API(`https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON`)를 호출하여 환율 데이터를 가져오고, Databricks Delta 테이블로 저장하는 노트북입니다.

### API 요청 변수
| 변수명 | 설명 | 예시 |
|---|---|---|
| `authkey` | 인증키 (API 등록 후 발급) | `AUTHKEY1234567890` |
| `searchdate` | 조회일자 (YYYYMMDD) | `20180102` |
| `data` | 데이터 구분 (AP01: 일반환율, AP02: 연간환율) | `AP01` |

### API 응답 필드
| 필드명 | 설명 |
|---|---|
| `result` | 결과 (1: 정상, 2: 데이터 없음) |
| `cur_unit` | 통화코드 (예: USD, EUR) |
| `cur_nm` | 통화명 (한글) |
| `ttb` | 전신환매도율 |
| `tts` | 전신환매입율 |
| `deal_bas_r` | 매매기준율 |
| `bkpr` | 장부가격 |
| `yy_efee_r` | 년환가수수료율 |
| `ten_dd_efee_r` | 10일환가수수료율 |
| `kftc_bkpr` | 서울외국환중개장부가격 |
| `kftc_deal_bas_r` | 서울외국환중개매매기준율 |
| `kftc_buy_kftc_bkpr` | 서울외국환중개 매전환매도 |
| `kftc_sell_kftc_bkpr` | 서울외국환중개 매전환매입 |

In [0]:
%skip

###################################################
# < workspace.default.authkey 에  authkey_sec 생성>

# catalog> workspace> default   에서   Create버튼 -> Secret 생성
 
###################################################


# OR


#####################################
# <한번만 생성 후 skip 처리 >
# 인증key 넣고 생성한후 
# 한번만 생성되면 되므로 skip 처리함
#####################################

%python
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import Secret

w = WorkspaceClient()
w.secrets_uc.create_secret(
    secret=Secret(
        name="authkey",
        catalog_name="workspace",
        schema_name="default",
        value="------------"  #### 인증key 넣고 생성한후 지움
    )
)
print("✅ Secret 생성 완료: workspace.default.authkey")



In [0]:
# ============================================
# 1단계: API 호출하여 환율 데이터 가져오기
# ============================================

import requests
import json
from datetime import datetime

# Unity Catalog Secret에서 인증키 읽어오기 (코드에 인증키가 노출되지 않음)
# dbutils.secrets.get()은 값을 직접 출력해도 ***로 마스킹됨
# 2번 셀에서 w.secrets_uc.create_secret()으로 생성한 workspace.default.authkey 를 가져옴
AUTH_KEY = dbutils.secrets.get(catalog="workspace", schema="default", key="authkey") # 2번 cell에서 생성한 secret키
# AUTH_KEY = dbutils.secrets.get(catalog="workspace", schema="default", key="authkey_sec") # catalog 에서 create 버튼으로 생성한 secret키

# 위젯에서 조회일자 읽어오기 (비어있으면 오늘 날짜 사용)
dbutils.widgets.text("search_date", "")
search_date = dbutils.widgets.get("search_date")
if not search_date:
    search_date = datetime.now().strftime("%Y-%m-%d")

# API 요청 URL 및 파라미터 설정
# searchdate: 조회일자
# data: AP01 (일반환율)
url = "https://oapi.koreaexim.go.kr/site/program/financial/exchangeJSON"
params = {
    "authkey": AUTH_KEY,   
    "searchdate": search_date,
    "data": "AP01"
}

# API 호출 (GET 방식)
response = requests.get(url, params=params)

# 응답 코드 확인 (200이면 정상)
print(f"HTTP 응답 코드: {response.status_code}")

# JSON 응답을 파이썬 리스트(배열)로 변환
exchange_data = response.json()

# 결과 출력
print(f"\n조회일자: {search_date}")
if exchange_data:
    result_code = exchange_data[0].get('result', 'N/A')
    print(f"API 결과 코드: {result_code} (1=성공, 2=데이터없음)")
    print(f"수신 통화 수: {len(exchange_data)} 개")
    print("\n첫 번째 데이터 샘플:")
    print(json.dumps(exchange_data[0], indent=2, ensure_ascii=False))
else:
    print("데이터가 없습니다. AUTH_KEY와 날짜를 확인하세요.")

In [0]:
# ============================================
# 2단계: JSON 데이터를 Spark DataFrame으로 변환
# ============================================
# PySpark는 대용량 데이터를 분산 처리하는 Databricks의 핵심 엔진입니다.
# DataFrame = SQL의 테이블과 같은 개념 (행과 열 구조의 데이터)
# 아래는 PySpark의 타입들인데, 여기서는 모든 환율 필드가 문자열로 들어오기 때문에 StringType을 사용합니다.

from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lit

# 스키마 정의 (API 응답 필드에 맞춤)
# SQL로 치면 CREATE TABLE의 컬럼 정의와 같습니다.
schema = StructType([
    StructField("result", StringType(), True),             # 결과 (1=정상, 2=데이터없음)
    StructField("cur_unit", StringType(), True),           # 통화코드 (USD, EUR 등)
    StructField("cur_nm", StringType(), True),             # 통화명 (한글)
    StructField("ttb", StringType(), True),                # 전신환매도율
    StructField("tts", StringType(), True),                # 전신환매입율
    StructField("deal_bas_r", StringType(), True),         # 매매기준율
    StructField("bkpr", StringType(), True),               # 장부가격
    StructField("yy_efee_r", StringType(), True),          # 년환가수수료율
    StructField("ten_dd_efee_r", StringType(), True),      # 10일환가수수료율
    StructField("kftc_bkpr", StringType(), True),          # 서울외국환중개장부가격
    StructField("kftc_deal_bas_r", StringType(), True),    # 서울외국환중개매매기준율
    StructField("kftc_buy_kftc_bkpr", StringType(), True), # 서울외국환중개 매전환매도
    StructField("kftc_sell_kftc_bkpr", StringType(), True),# 서울외국환중개 매전환매입
    # StructField("cur_nm", StringType(), True),             # 통화명 (한글) ----> 위로 옮김
])

# JSON 데이터(파이썬 리스트)를 Spark DataFrame으로 생성
# spark.createDataFrame() = 파이썬 데이터를 Spark 테이블 구조로 변환
df = spark.createDataFrame(exchange_data, schema=schema)

# 수집일자 컬럼 추가 (나중에 언제 수집한 데이터인지 추적하기 위해)
# lit() = 모든 행에 동일한 값을 넣는 함수 (SQL의 리터럴 값과 같음)
df = df.withColumn("search_date", lit(search_date))

# DataFrame 미리보기 (SQL의 SELECT * FROM ... LIMIT 10 과 같음)
# display()는 Databricks에서 데이터를 표 형태로 예쁘게 보여주는 함수
display(df)

In [0]:
# ============================================
# 3단계: Delta 테이블로 저장
# ============================================
# Delta 테이블 = Databricks에서 데이터를 영구적으로 저장하는 테이블
# SQL로 치면 CREATE TABLE ... AS SELECT ... 와 같습니다.


table_name = "workspace.default.exchange_rate_bronze"

# search_date별로 overwrite (과거 조회일자 환율 보존)
# replaceWhere: 해당 search_date의 데이터만 교체하고 나머지 날짜 데이터는 유지
# 같은 날짜를 다시 조회하면 해당 날짜 데이터만 새것으로 교체됨
if spark.catalog.tableExists(table_name):
    df.write \
        .mode("overwrite") \
        .option("replaceWhere", f"search_date = '{search_date}'") \
        .saveAsTable(table_name)
else:
    # 첫 실행 시 테이블 생성
    df.write \
        .mode("overwrite") \
        .saveAsTable(table_name)

print(f"✅ 테이블 {table_name} 에 search_date={search_date} 데이터 {df.count()}건이 저장/갱신되었습니다.")


In [0]:
%sql

-- 저장된 테이블 확인 (SQL)
SELECT search_date, * FROM workspace.default.exchange_rate_bronze  -- where search_date='20261012'
order by search_date

In [0]:
%sql
SELECT search_date, count(1) FROM workspace.default.exchange_rate_bronze group by search_date

-- delete FROM workspace.default.exchange_rate_bronze 

In [0]:
%skip

#############################
# 2 번 cell 로  변경함
# 2 번 cell 로  변경함
# 2 번 cell 로  변경함
#############################


%python
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# 1단계: Secret Scope 생성
# w.secrets.create_scope(scope="exchange_api_scope")

# 2단계: Secret 저장
# 위젯에서 authkey 값 읽어오기 (노트북 상단 위젯에 인증키 입력)
dbutils.widgets.text("authkey", "")
authkey_value = dbutils.widgets.get("authkey")

# 위젯에 입력된 값을 Secret으로 저장
w.secrets.put_secret(scope="exchange_api_scope", key="authkey", string_value=authkey_value)

print("Secret scope 생성 완료!")
print(authkey_value)

